In [1]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader

In [2]:
torch.manual_seed(42)

X = torch.randn(1000, 10)

y = (
    2 * X[:, 0]
    + X[:, 1]
    - X[:, 2]
    > 0
).long()

In [3]:
X_train = X[:800]
y_train = y[:800]

X_test = X[800:]
y_test = y[800:]

In [4]:
train_dataset = TensorDataset(
    X_train,
    y_train
)

In [5]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

In [6]:
class MLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(10, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 2)
        )

    def forward(self, x):
        return self.network(x)

In [7]:
model = MLP()

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [8]:
epochs = 30

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        # Forward
        logits = model(X_batch)

        # Loss
        loss = loss_fn(
            logits,
            y_batch
        )

        # Clear gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Parameter update
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch+1:<2}: | "
        f"Loss = {avg_loss:.4f}"
    )

Epoch 1 : | Loss = 0.6913
Epoch 2 : | Loss = 0.6031
Epoch 3 : | Loss = 0.4004
Epoch 4 : | Loss = 0.2002
Epoch 5 : | Loss = 0.1096
Epoch 6 : | Loss = 0.0746
Epoch 7 : | Loss = 0.0560
Epoch 8 : | Loss = 0.0439
Epoch 9 : | Loss = 0.0343
Epoch 10: | Loss = 0.0289
Epoch 11: | Loss = 0.0240
Epoch 12: | Loss = 0.0198
Epoch 13: | Loss = 0.0160
Epoch 14: | Loss = 0.0138
Epoch 15: | Loss = 0.0121
Epoch 16: | Loss = 0.0110
Epoch 17: | Loss = 0.0092
Epoch 18: | Loss = 0.0086
Epoch 19: | Loss = 0.0070
Epoch 20: | Loss = 0.0054
Epoch 21: | Loss = 0.0050
Epoch 22: | Loss = 0.0043
Epoch 23: | Loss = 0.0042
Epoch 24: | Loss = 0.0036
Epoch 25: | Loss = 0.0028
Epoch 26: | Loss = 0.0026
Epoch 27: | Loss = 0.0024
Epoch 28: | Loss = 0.0025
Epoch 29: | Loss = 0.0023
Epoch 30: | Loss = 0.0017


In [9]:
model.eval()

with torch.no_grad():

    logits = model(X_test)

    predictions = torch.argmax(
        logits,
        dim=1
    )

    accuracy = (
        predictions == y_test
    ).float().mean()

print(
    f"Test Accuracy: {accuracy.item():.4f}"
)

Test Accuracy: 0.9750


# 📘 Module 15.5 — Multi-Layer Perceptron (MLP) — Notes

## 1. What is an MLP? 🔥🔥🔥

**MLP = Multi-Layer Perceptron.**

An MLP is a **feed-forward neural network** consisting of multiple fully connected layers.

```text
Input
  ↓
Hidden Layer 1
  ↓
Hidden Layer 2
  ↓
   ...
  ↓
Output Layer
```

Example:

```text
10 → 64 → 32 → 16 → 3
```

Meaning:

* 10 input features
* 64 neurons
* 32 neurons
* 16 neurons
* 3 output neurons/classes

### Core idea

> **An MLP learns increasingly useful representations by applying multiple nonlinear transformations.**

---

# 2. Basic MLP Mathematics 🔥🔥🔥

A dense layer performs:

$$
z=Wx+b
$$

Then an activation is applied:

$$
a=f(z)
$$

For multiple layers:

$$
a^{[1]}=f(W^{[1]}x+b^{[1]})
$$

$$
a^{[2]}=f(W^{[2]}a^{[1]}+b^{[2]})
$$

$$
\hat y=f(W^{[3]}a^{[2]}+b^{[3]})
$$

Thus, an MLP repeatedly transforms the representation.

---

# 3. Why Multiple Layers? 🔥🔥🔥

A single linear layer can only represent a linear transformation.

If we stack only linear layers:

```text
Linear → Linear → Linear
```

they can mathematically collapse into another linear transformation:

$$
W_3(W_2(W_1x+b_1)+b_2)+b_3
$$

which can be represented as:

$$
Wx+b
$$

Therefore, simply adding more linear layers does **not** provide the desired nonlinear modeling capability.

### Solution

Add nonlinear activations:

```text
Linear
 ↓
ReLU
 ↓
Linear
 ↓
ReLU
 ↓
Linear
```

### Key memory

> **Layers provide transformations; activation functions provide nonlinearity.**

---

# 4. MLP Architecture

Example:

```text
Input
10 features
   ↓
Linear(10 → 64)
   ↓
ReLU
   ↓
Linear(64 → 32)
   ↓
ReLU
   ↓
Linear(32 → 16)
   ↓
ReLU
   ↓
Linear(16 → 3)
   ↓
3 Output Logits
```

For multiclass classification:

$$
\text{Output neurons}=\text{Number of classes}
$$

---

# 5. Hidden Layers and Representation Learning 🔥

The layers can progressively transform raw features into more useful representations.

Conceptually:

```text
Raw Input
    ↓
Simple Patterns
    ↓
Intermediate Patterns
    ↓
Higher-Level Patterns
    ↓
Prediction
```

This is called **hierarchical representation learning**.

The network learns these representations automatically through training.

---

# 6. MLP vs Perceptron

| Feature                  | Perceptron                 | MLP              |
| ------------------------ | -------------------------- | ---------------- |
| Architecture             | Single computational layer | Multiple layers  |
| Hidden layers            | ❌                          | ✅                |
| Activation               | Classical step             | ReLU/GELU/etc.   |
| Decision boundary        | Linear                     | Can be nonlinear |
| XOR                      | ❌                          | ✅                |
| Modern backprop training | Not typical                | Standard         |

### Architecture

**Perceptron:**

```text
Input → Output
```

**MLP:**

```text
Input → Hidden → Hidden → Output
```

---

# 7. PyTorch MLP Implementation 🔥🔥🔥

```python
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
```

### Model

```python
class MLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(10, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 2)
        )

    def forward(self, x):
        return self.network(x)
```

Architecture:

```text
10 → 64 → 32 → 16 → 2
```

The final layer produces **2 raw logits** for the two classes.

---

# 8. Loss and Optimizer

```python
model = MLP()

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)
```

For multiclass classification:

```text
MLP
 ↓
K raw logits
 ↓
CrossEntropyLoss
```

No Softmax is normally placed before `CrossEntropyLoss`.

---

# 9. MLP Training Loop 🔥🔥🔥

```python
for epoch in range(epochs):

    model.train()

    for X_batch, y_batch in train_loader:

        # Forward
        logits = model(X_batch)

        # Loss
        loss = loss_fn(logits, y_batch)

        # Clear old gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update parameters
        optimizer.step()
```

### Complete training pipeline

```text
Input
  ↓
Forward Pass
  ↓
Logits
  ↓
Loss
  ↓
Backward()
  ↓
Gradients
  ↓
Optimizer
  ↓
Updated Parameters
```

This is the same fundamental training pipeline you've already learned.

---

# 10. Tensor Shapes 🔥🔥🔥

Suppose:

* Batch size = 32
* Input features = 10
* Output classes = 2

Input:

```text
X → (32, 10)
```

First layer:

```text
(32,10)
   ↓
Linear(10,64)
   ↓
(32,64)
```

Second:

```text
(32,64)
   ↓
Linear(64,32)
   ↓
(32,32)
```

Third:

```text
(32,32)
   ↓
Linear(32,16)
   ↓
(32,16)
```

Output:

```text
(32,16)
   ↓
Linear(16,2)
   ↓
(32,2)
```

Target:

```text
y → (32,)
```

### Memory

$$
\boxed{\text{Input}=(B,\text{features})}
$$

$$
\boxed{\text{Output}=(B,\text{classes})}
$$

$$
\boxed{\text{Target}=(B,)}
$$

---

# 11. Parameter Count 🔥🔥🔥

For a fully connected layer:

$$
\text{Parameters}
=
n_{in}n_{out}+n_{out}
$$

because:

$$
\text{Weights}=n_{in}n_{out}
$$

and

$$
\text{Biases}=n_{out}
$$

### Example

For:

```text
10 → 64
```

$$
10(64)+64=704
$$

For:

```text
64 → 32
```

$$
64(32)+32=2080
$$

For:

```text
32 → 16
```

$$
32(16)+16=528
$$

For:

```text
16 → 2
```

$$
16(2)+2=34
$$

Total:

$$
704+2080+528+34=3346
$$

So:

> **The MLP has 3,346 trainable parameters.**

---

# 12. Why Parameter Count Matters

Increasing parameters can increase:

* Model capacity
* Memory usage
* Computation
* Ability to represent complex functions

But excessive capacity can also increase **overfitting risk**.

Therefore:

> **More parameters does not automatically mean a better model.**

---

# 13. Width vs Depth 🔥

### Width

Number of neurons in a layer.

```text
10 → 128 → 64 → 2
      ↑
    Width
```

### Depth

Number of successive computational layers.

```text
Input
 ↓
Layer 1
 ↓
Layer 2
 ↓
Layer 3
 ↓
Output
```

Increasing width:

```text
10 → 64 → 2
```

to:

```text
10 → 256 → 2
```

Increasing depth:

```text
10 → 64 → 2
```

to:

```text
10 → 64 → 32 → 16 → 2
```

---

# 14. MLP for Different Tasks 🔥🔥🔥

The hidden architecture can be similar, but the **output layer + loss** depend on the task.

### Regression

```text
MLP
 ↓
1 output
 ↓
Linear
 ↓
MSE / MAE / Huber
```

Example:

```python
nn.Linear(16, 1)
```

---

### Binary Classification

```text
MLP
 ↓
1 raw logit
 ↓
BCEWithLogitsLoss
```

```python
nn.Linear(16, 1)
```

---

### Multiclass Classification

```text
MLP
 ↓
K raw logits
 ↓
CrossEntropyLoss
```

```python
nn.Linear(16, K)
```

---

# 15. `nn.Sequential`

For a simple straight-line MLP:

```python
self.network = nn.Sequential(
    nn.Linear(10, 64),
    nn.ReLU(),

    nn.Linear(64, 32),
    nn.ReLU(),

    nn.Linear(32, 16),
    nn.ReLU(),

    nn.Linear(16, 2)
)
```

`nn.Sequential` is convenient when the data simply flows:

```text
Layer 1
 ↓
Layer 2
 ↓
Layer 3
 ↓
Layer 4
```

For architectures involving branches, skip/residual connections, multiple inputs, or custom operations, a custom `nn.Module` is more appropriate.

---

# 16. MLP vs Other Architectures

| Architecture    | Main structure   | Typical use                  |
| --------------- | ---------------- | ---------------------------- |
| **MLP**         | Fully connected  | Tabular/features             |
| **CNN**         | Convolution      | Images/spatial data          |
| **RNN**         | Recurrence       | Sequential data              |
| **LSTM/GRU**    | Gated recurrence | Sequences/time series        |
| **Transformer** | Attention        | Language, vision, multimodal |

### Important idea

An MLP treats an input primarily as a **feature vector**.

For example, a `28 × 28` image can be flattened:

$$
28\times28=784
$$

and given to an MLP as:

```text
784 features
```

But the MLP doesn't explicitly exploit the image's local spatial structure like a CNN does.

---

# 17. Common Mistakes 🚨

### ❌ No nonlinear activation

```python
nn.Linear(10, 64)
nn.Linear(64, 32)
nn.Linear(32, 2)
```

Multiple linear layers alone remain effectively linear.

---

### ❌ Softmax before `CrossEntropyLoss`

Normally:

```text
Linear → CrossEntropyLoss
```

not:

```text
Linear → Softmax → CrossEntropyLoss
```

---

### ❌ Wrong input size

If each sample has 10 features:

```python
nn.Linear(10, 64)
```

---

### ❌ Wrong output size

If there are 5 classes:

```python
nn.Linear(16, 5)
```

---

### ❌ Making the network unnecessarily huge

A larger network isn't automatically better.

It can mean:

* more computation
* more memory
* greater overfitting risk

---

# 18. Industry Relevance 💼

MLPs are commonly useful for:

* Tabular data
* Numerical feature prediction
* Classification
* Regression
* Baseline models
* Small/medium neural networks
* Feature-based prediction systems

MLP-like **feed-forward blocks** also appear inside larger architectures.

For example, Transformers contain feed-forward/MLP sublayers.

---

# 🎯 Interview / GATE Points

### What is an MLP?

A feed-forward neural network composed primarily of multiple fully connected layers with nonlinear activations.

### Why do hidden layers need activation functions?

To introduce nonlinearity and allow the network to model nonlinear relationships.

### Parameter count of a Dense layer?

$$
n_{in}n_{out}+n_{out}
$$

### What determines output neurons?

The task:

* Regression → usually 1
* Binary → usually 1 logit
* Multiclass → K logits

### What is the training pipeline?

```text
Forward
 ↓
Loss
 ↓
Backward
 ↓
Gradients
 ↓
Optimizer
 ↓
Parameter Update
```

---

# 🧠 Final Cheat Sheet

```text
                    MLP
                     │
                     ↓
               Input Features
                     │
                     ↓
              Linear + ReLU
                     │
                     ↓
              Linear + ReLU
                     │
                     ↓
              Linear + ReLU
                     │
                     ↓
                Output Layer
                     │
                     ↓
                  Logits
                     │
                     ↓
                   Loss
                     │
                     ↓
                Backprop
                     │
                     ↓
                 Optimizer
                     │
                     ↓
              Updated Weights
```

### 🔥 Remember these 6 points

1. **MLP = multiple fully connected layers.**
2. **Hidden activations provide nonlinearity.**
3. **MLPs learn hierarchical representations.**
4. **Dense parameters = `in × out + out`.**
5. **Output layer depends on the task.**
6. **Training remains Forward → Loss → Backward → Update.**

---